In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
from matplotlib.colors import LogNorm, Normalize
from IPython.display import display, Javascript

disable_js = """
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}
"""

def load_ipython_extension(ip):
    display(Javascript(disable_js))
    print ("autoscrolling long output is disabled")
    
load_ipython_extension(None)

In [ ]:
# Measured Simulation Output Bins
true_energies = np.array([0.005,0.015,0.025,0.035,0.045,0.055,0.065,0.075,0.085,
                        0.095, 0.105, 0.115, 0.125, 0.135, 0.145, 0.155, 0.165, 0.175, 0.185, 
                        0.195, 0.205, 0.215, 0.225, 0.235, 0.245, 0.260, 0.280, 0.300, 0.320, 
                        0.340, 0.360, 0.380, 0.400, 0.420, 0.440, 0.460, 0.480, 0.500, 0.520, 
                        0.540, 0.575, 0.625, 0.675, 0.725, 0.775, 0.825, 0.875, 0.925, 0.975, 
                        1.050, 1.150, 1.250, 1.350, 1.450, 1.550, 1.650, 1.750, 1.850, 1.950, 
                        2.050, 2.150, 2.250, 2.350, 2.450, 2.600, 2.800, 3.000, 3.200, 3.400, 
                        3.600, 3.800, 4.000, 4.200, 4.400, 4.600, 4.800, 5.000, 5.200, 5.400,
                        5.600, 5.800, 6.000, 6.200, 6.400, 6.600, 6.800, 7.000, 7.200, 7.400, 
                        7.750, 8.250, 8.750, 9.250, 9.750, 10.250, 10.750, 11.250, 11.750, 
                        12.250, 12.750, 13.250, 13.750, 14.250, 14.750, 15.500, 16.500, 17.500, 
                        18.500, 19.500, 20.500, 21.500, 22.500, 23.500, 24.500, 25.500, 26.500, 
                        27.500, 28.500, 29.500, 30.500, 31.500, 32.500, 33.500, 34.500, 35.500, 
                        36.500, 37.500, 38.500, 39.500, 40.500]) #MeV

In [ ]:
# NaI Reponse
NaI_files_unsorted = sorted(glob.glob("Hera Geant 4 Simulations/1e6_photon_sims/NaI*.out",recursive = True))
NaICountsDict = {}
binEdges = None

# Load event histograms into dict
# Read event energies from file names
for i, path in enumerate(NaI_files_unsorted):
    file = path.split('/')[2]
    energy = float(file.split('_')[1].split('MeV')[0]) # in MeVs
    events = np.loadtxt(path, skiprows=1, usecols=(1), dtype=float) / 1e6 # make units MeV (1e6 also num of photons)
    events = events[events > 0.005] #cuts out energies less than 300keV
    
    hist, binEdges_ = np.histogram(events, bins=true_energies, density=True)
    hist[np.isnan(hist)] = 0
    
    if binEdges is None:
        binEdges = binEdges_
    else:
        assert np.all(binEdges == binEdges_)
    
    NaICountsDict[energy] = hist

trueBinWidths = (true_energies - np.roll(true_energies, 1))[1:]
trueBinCenters = (true_energies + np.roll(true_energies, 1))[1:]/2

# Get count histograms
# Shape (True Energies, MeasuredBinEnergies)
NaIMatrix = np.zeros((trueBinCenters.size, trueBinCenters.size))

print(NaIMatrix.shape, trueBinWidths.shape)

keys = list(NaICountsDict.keys())
keys.sort()
for i, true_energy in enumerate(keys):
    NaIMatrix[i,:] = NaICountsDict[true_energy]


In [ ]:
labels = ['True Energies', 'Measured Energies']
z = np.sum(np.sum(NaIMatrix, axis=0) == 0)
print('Axis 0: {} '.format(labels[0] if z == 3 else labels[1]))
z = np.sum(np.sum(NaIMatrix, axis=1) == 0)
print('Axis 1: {} '.format(labels[0] if z == 3 else labels[1]))

In [ ]:
# TODO we need to test dividing out bins on both axes on the raw counts. We should be able to plot smooth histograms using different bins if
# this is handled correctly...

def norm_by_integral(vec, widths):
    out = vec / np.sum(widths * vec) # normalize by integral
    return vec / np.sum(vec) # normalize to sum to 1

NaIProbabilityMatrix = NaIMatrix.copy()
# NaIProbabilityMatrix = NaIProbabilityMatrix / trueBinWidths[:, None]
# NaIProbabilityMatrix = NaIProbabilityMatrix / trueBinWidths[None, :]
NaIProbabilityMatrix[np.isnan(NaIProbabilityMatrix)] = 0

NaIProbabilityMatrix /= (np.sum(NaIProbabilityMatrix, axis=1))[:,None]
NaIProbabilityMatrix[np.isnan(NaIProbabilityMatrix)] = 0

NaIProbability_test = np.sum(NaIProbabilityMatrix, axis=0)
NaIProbability_test[np.isnan(NaIProbability_test)] = 0

plt.figure(figsize=(12, 7), dpi=200)
plt.plot(trueBinCenters, trueBinWidths, 'y*', alpha=.05, label='bin widths', zorder=2)

#### Source Dist ######################################################################
sample_src = NaIProbability_test.copy()
# sample_src /= trueBinWidths
# sample_src *= trueBinWidths
sample_src /= np.sum(sample_src)
plt.plot(trueBinCenters, sample_src, 'g', linestyle='', marker='.', markersize=1,
         label='NaI P - Src Dist', alpha=.5)
print(sample_src[-10:])

#### Samples ##########################################################################
n = 1E7 # 10 million
energies_list = np.random.choice(trueBinCenters, p=sample_src, size=int(n))

# Hypothesis: If we are distribution of the sample in correct, we should be able to plot a density histogram with a
#             different set of bins and have it look visually the same...

# TODO README: difference in behavior between bins is probably because the test bins have only one
# count per bin at higher end, whereas the geatn4 bins still have many, that 
#
# TODO README: For Dr Holland: Summing over bins here is fine... In  general we multiply the reponse by the Dwyer distriubtion to get the
# "True distriubtion" I just left out that step here...

#### histograms #######################################################################
energies_hist, _ = np.histogram(energies_list, bins=true_energies, density=False)
energies_hist = energies_hist / np.sum(energies_hist)
print(np.sum(energies_hist), energies_hist.size)
print(np.sum(energies_hist!=0) / energies_hist.size)
plt.plot(trueBinCenters, energies_hist, 'red', marker='.', linestyle='', markersize=5,
         label='{:.0E} Photon sample density, bins=Geant4'.format(n), alpha=.9, zorder=1)

nbins = 1000
test_bins_edges = np.logspace(-2, 2, nbins)
test_bin_widths = np.diff(test_bins_edges)
test_bin_centers = test_bins_edges[:-1] + test_bin_widths
energies_hist, _ = np.histogram(energies_list, bins=test_bins_edges, density=False)
energies_hist = energies_hist / np.sum(energies_hist)
# energies_hist *= test_bin_widths # these dont work under any circumstances
# energies_hist /= test_bin_widths # these dont work under any circumstances
print(np.sum(energies_hist), energies_hist.size)
print(np.sum(energies_hist!=0) / energies_hist.size)
plt.plot(test_bin_centers, energies_hist, 'c', marker='.', linestyle='', markersize=10,
         label='{:.0E} Photon sample density, bins=test'.format(n), alpha=.9, zorder=0)

plt.yscale('log')
plt.xscale('log')
plt.xlabel('Measured Energy Spectrum [MeV]',fontsize=12)
plt.ylabel('Norm Meas. Spectrum',fontsize=12)
plt.legend()

In [ ]:
# Counts 
plt.figure(figsize=(6,4), dpi=200)
plt.imshow((NaIMatrix).T, cmap='plasma', norm=LogNorm(), origin='lower')

cb = plt.colorbar()
cb.ax.tick_params(labelsize=10)
cb.set_label(label='Counts',size=12)
plt.title('NaI Counts',fontsize=15)
plt.xlabel('True Energy Bins',fontsize=12)
plt.ylabel('Measured Energy Bins',fontsize=12)
plt.tick_params(labelsize=16)

# Divide into histogram probability
NaIProbabilityMatrix = NaIMatrix / (np.sum(NaIMatrix, axis=1))[:,None]
NaIProbabilityMatrix[np.isnan(NaIProbabilityMatrix)] = 0

plt.figure(figsize=(6,4), dpi=200)
plt.imshow((NaIProbabilityMatrix).T, cmap='plasma', norm=LogNorm(), origin='lower')
cb = plt.colorbar()
cb.ax.tick_params(labelsize=10)
cb.set_label(label='Measured Energy Probability',size=12)
plt.title('NaI Measured Bin Probability per True Energy',fontsize=10)
plt.xlabel('True Energy Bins',fontsize=12)
plt.ylabel('Measured Energy Bins',fontsize=12)
plt.tick_params(labelsize=16)
 
# # Divide into histogram probability density
# # NaIProbabilityDensityMatrix = NaIMatrix  / (np.sum(NaIMatrix * trueBinWidths, axis=1))[:,None] / trueBinWidths
# NaIProbabilityDensityMatrix = NaIMatrix  / (np.sum(NaIMatrix, axis=1))[:,None] / trueBinWidths
# NaIProbabilityDensityMatrix[np.isnan(NaIProbabilityDensityMatrix)] = 0
# 
# plt.figure(figsize=(6,4), dpi=200)
# plt.imshow((NaIProbabilityDensityMatrix).T, cmap='plasma', norm=LogNorm(), origin='lower')
# cb = plt.colorbar()
# cb.ax.tick_params(labelsize=10)
# cb.set_label(label='Measured Energy Probability Density',size=12)
# plt.title('NaI Measured Bin Probability per True Energy',fontsize=10)
# plt.xlabel('True Energy Bins',fontsize=12)
# plt.ylabel('Measured Energy Bins',fontsize=12)
# plt.tick_params(labelsize=16)
# 
# combined_nai_reponse = np.sum(NaIProbabilityDensityMatrix, axis=0)
# print(np.sum(combined_nai_reponse * trueBinWidths))

In [ ]:
plt.figure(figsize=(6,4), dpi=200)
plt.pcolormesh(true_energies, trueBinCenters, (NaIProbabilityMatrix).T,
               shading='auto',
               cmap='plasma',
               norm=LogNorm(),
               )

# Cbar left off because scale changed to interporlate log bins#
cb = plt.colorbar()
cb.ax.tick_params(labelsize=10)
cb.set_label(label='Measured Energy Probability Density',size=12)
plt.xlim(.1,35)
plt.ylim(.1,35)
plt.xscale('log')
plt.yscale('log')
plt.title('NaI Measured Energy Probability Density per True Energy',fontsize=10)
plt.xlabel('True Energy [MeV]',fontsize=12)
plt.ylabel('Measured Energy [MeV]',fontsize=12)
plt.tick_params(labelsize=16)


In [ ]:
NaIProbability = np.sum(NaIProbabilityMatrix, axis=0)

plt.figure(figsize=(8,6), dpi=200)
plt.plot(trueBinCenters, NaIProbability, 'g', label='NaI Probability', alpha=.5)
plt.plot(trueBinCenters, NaIProbability / trueBinWidths, 'g', label='NaI Density', alpha=.5)

n = 1E7 # 10 million
a = NaIProbability
a = NaIProbability / trueBinWidths
a = a / np.sum(a)

energies_list = np.random.choice(trueBinCenters, p=a, size=int(n))
energies_hist, _ = np.histogram(energies_list, bins=true_energies, density=True)
plt.plot(trueBinCenters, energies_hist / np.sum(energies_hist), 'pink', marker='.', linestyle='',
         label='{:.0E} Photon sample density, bins=Geant4'.format(n), alpha=.9)

# Hypothesis: If we are distribution of the sample in correct, wer should be able to plot a density histogram with a
#             different set of bins and have it look visually the same...

# TODO README unlike above, this trueBinWidths is squared because the density does not divide by same bin values...
energies_list = np.random.choice(trueBinCenters, p=a, size=int(n))
test_bins_edges = np.logspace(-2, 2, 129)
test_bin_widths = np.diff(test_bins_edges)
test_bin_centers = test_bins_edges[:-1] + test_bin_widths
energies_hist, _ = np.histogram(energies_list, bins=test_bins_edges, density=True)
plt.plot(test_bin_centers, energies_hist / np.sum(energies_hist), 'c.', label='{:.0E} Photon sample density, bins=test'.format(n), alpha=.9)

plt.yscale('log')
plt.xscale('log')
plt.xlabel('True Energy Spectrum [MeV]',fontsize=12)
plt.ylabel('Norm Meas. Spectrum',fontsize=12)
plt.legend()

In [ ]:
def TGF_Reader(file):
    gammalist = np.loadtxt(file)
    TGF_energy = gammalist[0:,0] #MeV
    #TGF_energy.fill(0.662) #to simulate a monoenergetic source of Cs137
    ZenithRad = gammalist[0:,2]
    ZenithDeg = ZenithRad * 180./np.pi
    return TGF_energy, ZenithDeg

def TGF_Filter(file):
    events, degrees = TGF_Reader(file)
    #events = np.delete(events,np.where(degrees<160.))
    #events = events[events > 0.005] #cuts out energies less than 10keV
    return events

#creates the TGF spectrum binned the same as the response matrix 
def TGF_spectrum(file):
    events = TGF_Filter(file)
    hist, binEdge = np.histogram(events, bins=true_energies)
    return hist
    

def ResponseSpectrum(file, matrix):
    TGF_Spectrum = TGF_spectrum(file)
    #TGF_diff = 1/binCenters * np.exp(-binCenters/7.3)
    #TGF_Spectrum = TGF_diff*binWidths
    output = TGF_Spectrum * 0
    #pdb.set_trace()
    for i in np.arange(len(matrix)):
        output = output + (matrix[i]*TGF_Spectrum[i])
    return output

TGF_file = "Hera Geant 4 Simulations/Dwyer_REAM_files/6km_downward_TGF/joeAltdown_6.txt"
TGF_Spectrum_density = TGF_spectrum(TGF_file)
# TGF_Spectrum_density = TGF_Spectrum_density / trueBinWidths
TGF_Spectrum_density = TGF_Spectrum_density / np.sum(TGF_Spectrum_density)

In [ ]:
dwyer = TGF_Spectrum_density 
dwyer = dwyer / trueBinWidths
# dwyer = dwyer / np.sum(dwyer)

combined_nai_reponse = np.sum(NaIProbabilityDensityMatrix, axis=0)
combined_nai_reponse = combined_nai_reponse / trueBinWidths
# combined_nai_reponse = combined_nai_reponse / np.sum(combined_nai_reponse)

NaI_Dwyer_Response = np.sum(NaIProbabilityDensityMatrix, axis=0)
NaI_Dwyer_Response_unnorm_copy = NaI_Dwyer_Response.copy() 
NaI_Dwyer_Response = NaI_Dwyer_Response / trueBinWidths ** 2 #TODO not the square here is required to make it smooth...
# NaI_Dwyer_Response = NaI_Dwyer_Response / np.sum(NaI_Dwyer_Response)

print(dwyer.shape, combined_nai_reponse.shape, NaI_Dwyer_Response.shape)

plt.figure(figsize=(6,3), dpi=200)
plt.plot(trueBinCenters[1:], dwyer[1:], 'y', label='Dwyer Spectrum Density', alpha=.75)
plt.plot(trueBinCenters, combined_nai_reponse, 'b', label='NaI Response Density', alpha=.75)
plt.plot(trueBinCenters[1:], NaI_Dwyer_Response[1:], 'g', label='NaI - Dwyer Combined Spectrum Density', alpha=.75)

plt.yscale('log')
plt.xscale('log')
plt.xlabel('True Energy Spectrum [MeV]',fontsize=12)
plt.ylabel('Norm Meas. Spectrum',fontsize=12)
# plt.xlim([10E-2, 100])
plt.legend()

In [ ]:
plt.figure(figsize=(8,6), dpi=200)
plt.plot(trueBinCenters[1:], NaI_Dwyer_Response[1:], 'g', label='NaI - Dwyer Combined Spectrum Density', alpha=.5)

n = 1E7 # 10 million
# a = NaI_Dwyer_Response_unnorm_copy / trueBinWidths
a = NaI_Dwyer_Response_unnorm_copy * trueBinWidths
a = a/np.sum(a)
energies_list = np.random.choice(trueBinCenters, p=a, size=int(n))
energies_hist, _ = np.histogram(energies_list, bins=true_energies, density=True)
plt.plot(trueBinCenters, energies_hist / np.sum(energies_hist), 'pink', marker='.', linestyle='',
         label='{:.0E} Photon sample density, bins=Geant4'.format(n), alpha=.9)

# Hypothesis: If we are distribution of the sample in correct, wer should be able to plot a density histogram with a
#             different set of bins and have it look visually the same...

# TODO README unlike above, this trueBinWidths is squared because the density does not divide by same bin values...
# a = NaI_Dwyer_Response_unnorm_copy  / trueBinWidths ** 2
# a = NaI_Dwyer_Response_unnorm_copy * trueBinWidths ** 2
a = NaI_Dwyer_Response_unnorm_copy  / trueBinWidths
a = a/np.sum(a)
energies_list = np.random.choice(trueBinCenters, p=a, size=int(n))
test_bins_edges = np.logspace(-2, 2, 1001)
test_bin_widths = np.diff(test_bins_edges)
test_bin_centers = test_bins_edges[:-1] + test_bin_widths
energies_hist, _ = np.histogram(energies_list, bins=test_bins_edges, density=True)
energies_hist = energies_hist / test_bin_widths
energies_hist = energies_hist / np.sum(energies_hist)
plt.plot(test_bin_centers, energies_hist / np.sum(energies_hist), 'c.', label='{:.0E} Photon sample density, bins=test'.format(n), alpha=.9)

diff = np.diff(trueBinWidths)
mask = diff > 1E-5
plt.plot(trueBinCenters[:-1][mask], diff[mask], label='True (Geant4) Bin Diffs >> 0', marker='*', linestyle='', alpha=.25)
diff = np.diff(test_bin_widths)
plt.plot(test_bin_centers[:-1], diff, label='Test Bin Diffs', marker='*', linestyle='', alpha=.25)

plt.yscale('log')
plt.xscale('log')
plt.xlabel('True Energy Spectrum [MeV]',fontsize=12)
plt.ylabel('Norm Meas. Spectrum',fontsize=12)
plt.legend()